In [ ]:

# CELL 1: ENVIRONMENT SETUP AND DATASET DOWNLOAD

!pip -q install kagglehub scikit-learn seaborn

import os
import time
import random
import json
import copy
import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device information
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("EFFICIENTNET-B0 FEDERATED MAML RETRAINING")
print("=" * 65)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable. Enable a GPU runtime in Colab.")

# Download public Kaggle dataset
import kagglehub

dataset_path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

print("\nKaggle dataset downloaded to:")
print(dataset_path)

# Find the dataset's Training and Testing directories
DATASET_ROOT = None

for root, dirs, files in os.walk(dataset_path):
    if "Training" in dirs and "Testing" in dirs:
        DATASET_ROOT = root
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not locate Training and Testing directories. "
        "Send me the output and we will fix the dataset path."
    )

TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")
TEST_DIR = os.path.join(DATASET_ROOT, "Testing")

print("\nDataset root:", DATASET_ROOT)
print("Training directory exists:", os.path.isdir(TRAIN_DIR))
print("Testing directory exists:", os.path.isdir(TEST_DIR))

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

print("\nClass image counts:")

for split_dir in [TRAIN_DIR, TEST_DIR]:
    print("\n", os.path.basename(split_dir))

    for class_name in CLASS_NAMES:
        class_dir = os.path.join(split_dir, class_name)

        if not os.path.isdir(class_dir):
            raise FileNotFoundError(
                f"Missing class directory: {class_dir}"
            )

        count = sum(
            1 for name in os.listdir(class_dir)
            if name.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            )
        )

        print(f"  {class_name:12s}: {count}")

# Create persistent output directory
OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Brain_Tumor_FL_MAML/EfficientNetB0_Retraining"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nOutput directory:", OUTPUT_DIR)
print("\nCELL 1 COMPLETE")


In [ ]:

# CELL 2: DATA LOADERS AND FEDERATED CLIENT PARTITIONS

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import Counter

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_CLIENTS = 5

# ImageNet normalization for pretrained EfficientNet-B0
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

# Verify class ordering
CLASS_NAMES = train_dataset.classes

assert len(CLASS_NAMES) == NUM_CLASSES
assert train_dataset.classes == test_dataset.classes

print("Class-to-index mapping:")
print(train_dataset.class_to_idx)

print("\nTraining images:", len(train_dataset))
print("Testing images:", len(test_dataset))

# Create balanced, reproducible client partitions
rng = np.random.default_rng(SEED)
targets = np.array(train_dataset.targets)

client_indices = [[] for _ in range(NUM_CLIENTS)]

for class_idx in range(NUM_CLASSES):
    class_indices = np.where(targets == class_idx)[0]
    rng.shuffle(class_indices)

    splits = np.array_split(class_indices, NUM_CLIENTS)

    for client_id, split in enumerate(splits):
        client_indices[client_id].extend(
            split.tolist()
        )

# Shuffle within each client's partition
for indices in client_indices:
    rng.shuffle(indices)

# Create data loaders
NUM_WORKERS = 2

client_loaders = {}

for client_id, indices in enumerate(client_indices, start=1):
    subset = Subset(train_dataset, indices)

    client_loaders[client_id] = DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Verify per-client class balance
print("\nFEDERATED CLIENT DISTRIBUTION")
print("-" * 60)

for client_id, indices in enumerate(client_indices, start=1):
    counts = Counter(targets[indices].tolist())

    print(
        f"Client {client_id}: "
        f"{len(indices)} images | "
        + " | ".join(
            f"{CLASS_NAMES[k]}={counts[k]}"
            for k in range(NUM_CLASSES)
        )
    )

assert sum(len(x) for x in client_indices) == len(train_dataset)
assert len(set(i for part in client_indices for i in part)) == len(train_dataset)

print("\nTest batches:", len(test_loader))
print("CELL 2 COMPLETE")


In [ ]:

# CELL 3: EFFICIENTNET-B0 MODEL AND FOMAML FUNCTIONS

import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.func import functional_call
import time
import copy

# Federated training configuration
ROUNDS = 10
LOCAL_EPOCHS = 10
SELECTED_CLIENTS = 3
INNER_LR = 1e-4
OUTER_LR = 1e-4

# Fixed client schedule used in the previous experiment
CLIENT_SCHEDULE = {
    1: [1, 3, 5],
    2: [1, 2, 3],
    3: [1, 2, 3],
    4: [1, 3, 5],
    5: [1, 4, 5],
    6: [1, 2, 5],
    7: [1, 3, 5],
    8: [1, 2, 4],
    9: [1, 3, 4],
    10: [2, 4, 5]
}

def create_model():
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.DEFAULT
    )

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES
    )

    return model

# Initialize global model
global_model = create_model().to(DEVICE)

# Count parameters
total_parameters = sum(
    p.numel() for p in global_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in global_model.parameters()
    if p.requires_grad
)

model_size_mb = sum(
    p.numel() * p.element_size()
    for p in global_model.parameters()
) / (1024 ** 2)

print("EfficientNet-B0 initialized")
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Parameter size:", f"{model_size_mb:.2f} MB")
print("Device:", DEVICE)

# Verify functional_call compatibility
sample_parameters = dict(global_model.named_parameters())
sample_buffers = dict(global_model.named_buffers())

print("Functional parameter tensors:", len(sample_parameters))
print("Model output classes:", NUM_CLASSES)

# First-order MAML local adaptation
def maml_adapt(model, support_x, support_y, inner_lr=INNER_LR):
    model.train()

    params = {
        name: param.detach().clone().requires_grad_(True)
        for name, param in model.named_parameters()
    }

    buffers = {
        name: buffer.detach().clone()
        for name, buffer in model.named_buffers()
    }

    logits = functional_call(
        model, (params, buffers), (support_x,)
    )

    support_loss = F.cross_entropy(logits, support_y)

    grads = torch.autograd.grad(
        support_loss,
        tuple(params.values()),
        create_graph=False
    )

    adapted_params = {
        name: (param - inner_lr * grad).detach()
        for (name, param), grad in zip(params.items(), grads)
    }

    return adapted_params, buffers, support_loss.detach()

# Test forward pass
global_model.eval()

with torch.no_grad():
    sample_images, sample_labels = next(iter(test_loader))
    sample_images = sample_images[:2].to(DEVICE)
    sample_output = global_model(sample_images)

print("Test output shape:", tuple(sample_output.shape))
print("Sample labels:", sample_labels[:2].tolist())


In [ ]:
import os
import copy
import time
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch.func import functional_call

# Checkpoint directory on Google Drive
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(path, model, round_num=None, extra=None):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "round": round_num,
        "model_name": "EfficientNet-B0",
        "image_size": IMG_SIZE,
        "normalization_mean": [0.485, 0.456, 0.406],
        "normalization_std": [0.229, 0.224, 0.225],
        "seed": SEED,
        "extra": extra or {}
    }
    torch.save(checkpoint, path)

def clone_buffers(model):
    return {
        name: buffer.detach().clone()
        for name, buffer in model.named_buffers()
    }

def first_order_maml_step(model, optimizer, support_x, support_y,
                          query_x, query_y):
    """
    First-order MAML-style update:
    1. Compute support-set adaptation gradients.
    2. Adapt parameters using the inner learning rate.
    3. Compute query gradients at adapted parameters.
    4. Transfer those gradients to the original parameters.
    """
    model.train()
    optimizer.zero_grad(set_to_none=True)

    base_params = dict(model.named_parameters())
    support_params = {
        name: param.detach().clone().requires_grad_(True)
        for name, param in base_params.items()
    }
    support_buffers = clone_buffers(model)

    support_logits = functional_call(
        model, (support_params, support_buffers), (support_x,)
    )
    support_loss = F.cross_entropy(support_logits, support_y)

    support_grads = torch.autograd.grad(
        support_loss,
        tuple(support_params.values()),
        create_graph=False
    )

    adapted_params = {
        name: (
            param.detach() - INNER_LR * grad.detach()
        ).requires_grad_(True)
        for (name, param), grad in zip(
            support_params.items(), support_grads
        )
    }

    # Use fresh buffers for the query forward pass.
    query_buffers = clone_buffers(model)
    query_logits = functional_call(
        model, (adapted_params, query_buffers), (query_x,)
    )
    query_loss = F.cross_entropy(query_logits, query_y)

    query_grads = torch.autograd.grad(
        query_loss,
        tuple(adapted_params.values()),
        create_graph=False
    )

    # First-order approximation: transfer query gradients to base parameters.
    for (name, param), grad in zip(base_params.items(), query_grads):
        param.grad = grad.detach().clone()

    optimizer.step()

    return {
        "support_loss": float(support_loss.detach().item()),
        "query_loss": float(query_loss.detach().item())
    }

def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "labels": np.array(all_labels),
        "predictions": np.array(all_preds),
        "probabilities": np.array(all_probs)
    }

print("Training and evaluation functions are ready.")
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Client schedule:", CLIENT_SCHEDULE)

In [ ]:
# Smoke test: one first-order MAML update on Client 1

test_model = copy.deepcopy(global_model).to(DEVICE)
test_optimizer = torch.optim.Adam(
    test_model.parameters(),
    lr=OUTER_LR
)

images, labels = next(iter(client_loaders[1]))
images = images.to(DEVICE, non_blocking=True)
labels = labels.to(DEVICE, non_blocking=True)

# Split the batch into support and query sets
split_idx = images.size(0) // 2

support_x, query_x = images[:split_idx], images[split_idx:]
support_y, query_y = labels[:split_idx], labels[split_idx:]

# Check parameter values before the update
param_before = next(test_model.parameters()).detach().clone()

start_time = time.time()
smoke_result = first_order_maml_step(
    test_model,
    test_optimizer,
    support_x,
    support_y,
    query_x,
    query_y
)
elapsed = time.time() - start_time

param_after = next(test_model.parameters()).detach()
parameter_change = (param_after - param_before).abs().max().item()

print("Support loss:", round(smoke_result["support_loss"], 4))
print("Query loss:", round(smoke_result["query_loss"], 4))
print("Update time (seconds):", round(elapsed, 3))
print("Maximum parameter change:", parameter_change)
print("Gradient update successful:", parameter_change > 0)
print("GPU memory allocated (MB):",
      round(torch.cuda.memory_allocated() / (1024 ** 2), 2))

In [ ]:
import gc
import pandas as pd
import torch
import torch.nn.functional as F
import time
import os
import copy

history = []
training_start = time.time()

# Save the initialized model as a recovery checkpoint
save_checkpoint(
    os.path.join(CHECKPOINT_DIR, "initial_checkpoint.pth"),
    global_model,
    round_num=0
)

for round_num in range(1, ROUNDS + 1):
    round_start = time.time()
    selected_clients = CLIENT_SCHEDULE[round_num]

    print(f"\n{'=' * 65}")
    print(f"Round {round_num}/{ROUNDS} | Selected clients: {selected_clients}")
    print(f"{'=' * 65}")

    # Snapshot global model parameters for federated aggregation
    global_state = {
        name: tensor.detach().clone()
        for name, tensor in global_model.state_dict().items()
    }

    client_states = []
    client_sizes = []
    client_losses = []

    for client_id in selected_clients:
        client_start = time.time()

        local_model = copy.deepcopy(global_model).to(DEVICE)
        optimizer = torch.optim.Adam(
            local_model.parameters(),
            lr=OUTER_LR
        )

        support_loss_sum = 0.0
        query_loss_sum = 0.0
        update_count = 0

        for epoch in range(LOCAL_EPOCHS):
            local_model.train()

            for images, labels in client_loaders[client_id]:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                # Skip batches too small to form support and query sets
                if images.size(0) < 2:
                    continue

                split_idx = images.size(0) // 2
                support_x, query_x = images[:split_idx], images[split_idx:]
                support_y, query_y = labels[:split_idx], labels[split_idx:]

                losses = first_order_maml_step(
                    local_model,
                    optimizer,
                    support_x,
                    support_y,
                    query_x,
                    query_y
                )

                support_loss_sum += losses["support_loss"]
                query_loss_sum += losses["query_loss"]
                update_count += 1

        # Keep each client's locally trained parameters for aggregation
        client_states.append({
            name: tensor.detach().cpu().clone()
            for name, tensor in local_model.state_dict().items()
        })
        client_sizes.append(len(client_indices[client_id - 1]))

        mean_support_loss = support_loss_sum / max(update_count, 1)
        mean_query_loss = query_loss_sum / max(update_count, 1)
        client_losses.append(mean_query_loss)

        print(
            f"Client {client_id}: "
            f"support loss={mean_support_loss:.4f}, "
            f"query loss={mean_query_loss:.4f}, "
            f"time={(time.time() - client_start) / 60:.2f} min"
        )

        del local_model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

    # Federated weighted averaging (FedAvg) of client model states
    total_client_samples = sum(client_sizes)
    aggregated_state = {}

    for name, global_tensor in global_state.items():
        if torch.is_floating_point(global_tensor):
            aggregated_state[name] = sum(
                client_states[i][name].to(global_tensor.device)
                * (client_sizes[i] / total_client_samples)
                for i in range(len(client_states))
            )
        else:
            # For integer buffers such as BatchNorm counters, retain the
            # global value rather than averaging integer values.
            aggregated_state[name] = global_tensor

    global_model.load_state_dict(aggregated_state)

    # Evaluate the aggregated global model
    evaluation = evaluate_model(global_model, test_loader)

    round_time = time.time() - round_start
    record = {
        "round": round_num,
        "selected_clients": str(selected_clients),
        "mean_client_query_loss": float(np.mean(client_losses)),
        "test_loss": evaluation["loss"],
        "test_accuracy": evaluation["accuracy"],
        "round_time_minutes": round_time / 60
    }
    history.append(record)

    # Save a round-specific checkpoint and training history
    round_path = os.path.join(
        CHECKPOINT_DIR, f"efficientnet_b0_round_{round_num:02d}.pth"
    )
    save_checkpoint(
        round_path,
        global_model,
        round_num=round_num,
        extra=record
    )

    history_path = os.path.join(OUTPUT_DIR, "training_history.csv")
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(
        f"Round {round_num} summary | "
        f"Test accuracy: {evaluation['accuracy'] * 100:.2f}% | "
        f"Test loss: {evaluation['loss']:.4f} | "
        f"Round time: {round_time / 60:.2f} min"
    )
    print(f"Checkpoint saved: {round_path}")

# Save final checkpoint separately
final_checkpoint_path = os.path.join(
    OUTPUT_DIR, "efficientnet_b0_federated_maml_final.pth"
)
save_checkpoint(
    final_checkpoint_path,
    global_model,
    round_num=ROUNDS,
    extra={"training_history": history}
)

total_training_minutes = (time.time() - training_start) / 60

print("\nTraining finished.")
print(f"Total training time: {total_training_minutes:.2f} minutes")
print(f"Final checkpoint: {final_checkpoint_path}")
print(f"History CSV: {history_path}")

display(pd.DataFrame(history))

In [ ]:
import os
import torch

final_checkpoint_path = (
    "/content/drive/MyDrive/Brain_Tumor_FL_MAML/"
    "EfficientNetB0_Retraining/"
    "efficientnet_b0_federated_maml_final.pth"
)

assert os.path.isfile(final_checkpoint_path), (
    f"Checkpoint not found: {final_checkpoint_path}"
)

file_size_mb = os.path.getsize(final_checkpoint_path) / (1024 ** 2)

checkpoint = torch.load(
    final_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint verified successfully.")
print(f"File: {final_checkpoint_path}")
print(f"File size: {file_size_mb:.2f} MB")
print("Model:", checkpoint["model_name"])
print("Saved round:", checkpoint["round"])
print("Classes:", checkpoint["class_names"])
print("Number of state-dict entries:",
      len(checkpoint["model_state_dict"]))

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Reload the saved checkpoint
checkpoint = torch.load(
    final_checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

evaluation_model = models.efficientnet_b0(weights=None)
evaluation_model.classifier[1] = nn.Linear(
    evaluation_model.classifier[1].in_features,
    len(checkpoint["class_names"])
)
evaluation_model.load_state_dict(checkpoint["model_state_dict"])
evaluation_model = evaluation_model.to(DEVICE)
evaluation_model.eval()

# Run inference on the test dataset
evaluation = evaluate_model(evaluation_model, test_loader)

y_true = evaluation["labels"]
y_pred = evaluation["predictions"]
y_prob = evaluation["probabilities"]
class_names = checkpoint["class_names"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)
recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)
f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

# Macro one-vs-rest specificity
specificities = []
for i in range(len(class_names)):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificity_i = tn / (tn + fp) if (tn + fp) else 0.0
    specificities.append(specificity_i)

specificity = float(np.mean(specificities))

y_true_binary = label_binarize(
    y_true, classes=list(range(len(class_names)))
)
roc_auc = roc_auc_score(
    y_true_binary,
    y_prob,
    average="macro",
    multi_class="ovr"
)

metrics = {
    "model": "EfficientNet-B0",
    "checkpoint_round": checkpoint["round"],
    "test_samples": int(len(y_true)),
    "accuracy_percent": float(accuracy * 100),
    "macro_precision_percent": float(precision * 100),
    "macro_recall_percent": float(recall * 100),
    "macro_specificity_percent": float(specificity * 100),
    "macro_f1_percent": float(f1 * 100),
    "macro_roc_auc_percent": float(roc_auc * 100),
    "test_loss": float(evaluation["loss"])
}

print("FINAL CHECKPOINT EVALUATION")
print("-" * 45)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
    digits=4
))

print("Confusion matrix:")
print(cm)

# Save numerical results
metrics_path = os.path.join(
    OUTPUT_DIR, "efficientnet_b0_final_metrics.json"
)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
pd.DataFrame(report).transpose().to_csv(
    os.path.join(OUTPUT_DIR, "classification_report.csv")
)

np.save(
    os.path.join(OUTPUT_DIR, "confusion_matrix.npy"),
    cm
)

# Plot and save the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("EfficientNet-B0 — Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()

cm_path = os.path.join(
    OUTPUT_DIR, "efficientnet_b0_confusion_matrix.png"
)
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved metrics:", metrics_path)
print("Saved confusion matrix:", cm_path)
print("All evaluation outputs are in:", OUTPUT_DIR)

In [ ]:
import os
import json
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve,
    auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize

# ---------------------------------------------------------
# 1. Paths and configuration
# ---------------------------------------------------------
RESULTS_DIR = "/content/drive/MyDrive/Brain_Tumor_FL_MAML/Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_PREFIX = "efficientnet_b0"
NUM_CLASSES = len(CLASS_NAMES)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
FINAL_CHECKPOINT = os.path.join(
    OUTPUT_DIR, "efficientnet_b0_federated_maml_final.pth"
)
HISTORY_CSV = os.path.join(OUTPUT_DIR, "training_history.csv")

assert os.path.isfile(FINAL_CHECKPOINT), "Final checkpoint is missing."
assert os.path.isfile(HISTORY_CSV), "Training history CSV is missing."

history_df = pd.read_csv(HISTORY_CSV)
round_records = []
final_eval = None
final_cm = None
final_report = None

# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------
def load_round_model(checkpoint_path):
    ckpt = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        len(ckpt["class_names"])
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()
    return model, ckpt


def calculate_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(
        y_true, y_pred, labels=list(range(NUM_CLASSES))
    )

    per_class_specificity = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        per_class_specificity.append(
            tn / (tn + fp) if (tn + fp) else 0.0
        )

    y_binary = label_binarize(
        y_true, classes=list(range(NUM_CLASSES))
    )
    auc_value = roc_auc_score(
        y_binary, y_prob, average="macro"
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "specificity": float(np.mean(per_class_specificity)),
        "f1_score": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "roc_auc": auc_value,
        "confusion_matrix": cm
    }


def save_plot(filename):
    path = os.path.join(RESULTS_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", path)


# ---------------------------------------------------------
# 3. Evaluate every saved round checkpoint
# ---------------------------------------------------------
print("Evaluating saved round checkpoints...")

for round_num in range(1, ROUNDS + 1):
    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"efficientnet_b0_round_{round_num:02d}.pth"
    )
    assert os.path.isfile(checkpoint_path), (
        f"Missing checkpoint for round {round_num}: {checkpoint_path}"
    )

    round_model, round_ckpt = load_round_model(checkpoint_path)
    round_eval = evaluate_model(round_model, test_loader)

    round_metrics = calculate_metrics(
        round_eval["labels"],
        round_eval["predictions"],
        round_eval["probabilities"]
    )

    matching_history = history_df.loc[
        history_df["round"] == round_num
    ].iloc[0]

    record = {
        "round": round_num,
        "test_loss": float(round_eval["loss"]),
        "accuracy": float(round_metrics["accuracy"]),
        "precision": float(round_metrics["precision"]),
        "recall": float(round_metrics["recall"]),
        "specificity": float(round_metrics["specificity"]),
        "f1_score": float(round_metrics["f1_score"]),
        "roc_auc": float(round_metrics["roc_auc"]),
        "mean_client_query_loss": float(
            matching_history["mean_client_query_loss"]
        ),
        "round_time_minutes": float(
            matching_history["round_time_minutes"]
        ),
        "selected_clients": str(matching_history["selected_clients"])
    }
    round_records.append(record)

    if round_num == ROUNDS:
        final_eval = round_eval
        final_cm = round_metrics["confusion_matrix"]
        final_report = classification_report(
            round_eval["labels"],
            round_eval["predictions"],
            labels=list(range(NUM_CLASSES)),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
        final_y_true = round_eval["labels"]
        final_y_pred = round_eval["predictions"]
        final_y_prob = round_eval["probabilities"]

    print(
        f"Round {round_num:02d}: "
        f"accuracy={round_metrics['accuracy'] * 100:.2f}% | "
        f"loss={round_eval['loss']:.4f} | "
        f"macro-F1={round_metrics['f1_score'] * 100:.2f}%"
    )

metrics_df = pd.DataFrame(round_records)
metrics_df.to_csv(
    os.path.join(RESULTS_DIR, f"{MODEL_PREFIX}_metrics_by_round.csv"),
    index=False
)

# ---------------------------------------------------------
# 4. Graph 1: Global test accuracy vs round
# ---------------------------------------------------------
plt.figure(figsize=(9, 5))
plt.plot(
    metrics_df["round"],
    metrics_df["accuracy"] * 100,
    marker="o",
    linewidth=2,
    label="Test accuracy"
)
plt.xticks(range(1, ROUNDS + 1))
plt.xlabel("Federated round")
plt.ylabel("Accuracy (%)")
plt.title("EfficientNet-B0: Global Test Accuracy vs Round")
plt.grid(True, alpha=0.3)
plt.legend()
save_plot(f"{MODEL_PREFIX}_global_accuracy_vs_round.png")

# ---------------------------------------------------------
# 5. Graph 2: NEW LOSS CURVE
# ---------------------------------------------------------
plt.figure(figsize=(9, 5))
plt.plot(
    metrics_df["round"],
    metrics_df["mean_client_query_loss"],
    marker="o",
    linewidth=2,
    label="Mean client query loss"
)
plt.plot(
    metrics_df["round"],
    metrics_df["test_loss"],
    marker="s",
    linewidth=2,
    label="Global test loss"
)
plt.xticks(range(1, ROUNDS + 1))
plt.xlabel("Federated round")
plt.ylabel("Cross-entropy loss")
plt.title("EfficientNet-B0: Loss Curve")
plt.grid(True, alpha=0.3)
plt.legend()
save_plot(f"{MODEL_PREFIX}_loss_curve.png")

# ---------------------------------------------------------
# 6. Graph 3: Classification metrics vs round
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
for metric, label in [
    ("precision", "Macro Precision"),
    ("recall", "Macro Recall"),
    ("specificity", "Macro Specificity"),
    ("f1_score", "Macro F1-score")
]:
    plt.plot(
        metrics_df["round"],
        metrics_df[metric] * 100,
        marker="o",
        linewidth=2,
        label=label
    )

plt.xticks(range(1, ROUNDS + 1))
plt.xlabel("Federated round")
plt.ylabel("Score (%)")
plt.title("EfficientNet-B0: Classification Metrics vs Round")
plt.grid(True, alpha=0.3)
plt.legend()
save_plot(f"{MODEL_PREFIX}_classification_metrics_vs_round.png")

# ---------------------------------------------------------
# 7. Graph 4: Final classification metrics
# ---------------------------------------------------------
final_row = metrics_df.iloc[-1]
final_metric_names = [
    "accuracy", "precision", "recall",
    "specificity", "f1_score", "roc_auc"
]
final_metric_labels = [
    "Accuracy", "Macro Precision", "Macro Recall",
    "Macro Specificity", "Macro F1", "Macro ROC-AUC"
]
final_values = [final_row[m] * 100 for m in final_metric_names]

plt.figure(figsize=(10, 5))
bars = plt.bar(final_metric_labels, final_values)
plt.ylim(0, 105)
plt.ylabel("Score (%)")
plt.title("EfficientNet-B0: Final Classification Metrics")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, final_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

save_plot(f"{MODEL_PREFIX}_final_metrics.png")

# ---------------------------------------------------------
# 8. Graph 5: Final confusion matrix
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.heatmap(
    final_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("EfficientNet-B0: Confusion Matrix")
save_plot(f"{MODEL_PREFIX}_confusion_matrix.png")

# ---------------------------------------------------------
# 9. Graph 6: Multiclass ROC curves
# ---------------------------------------------------------
y_true_binary = label_binarize(
    final_y_true, classes=list(range(NUM_CLASSES))
)

plt.figure(figsize=(8, 6))
for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(
        y_true_binary[:, i], final_y_prob[:, i]
    )
    class_auc = auc(fpr, tpr)
    plt.plot(
        fpr, tpr, linewidth=2,
        label=f"{class_name} (AUC = {class_auc:.4f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("EfficientNet-B0: One-vs-Rest ROC Curves")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
save_plot(f"{MODEL_PREFIX}_roc_curves.png")

# ---------------------------------------------------------
# 10. Graph 7: Cumulative estimated communication
# ---------------------------------------------------------
# This is an estimate, not measured network traffic.
# Assumes each selected client downloads and uploads one model state.
final_model, final_ckpt = load_round_model(FINAL_CHECKPOINT)

state_size_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in final_model.state_dict().values()
)
state_size_mb = state_size_bytes / (1024 ** 2)

selected_client_counts = metrics_df["selected_clients"].apply(
    lambda x: len(json.loads(x.replace("'", '"')))
)
estimated_round_mb = (
    2 * selected_client_counts * state_size_mb
)
cumulative_communication_mb = estimated_round_mb.cumsum()

plt.figure(figsize=(9, 5))
plt.plot(
    metrics_df["round"],
    cumulative_communication_mb,
    marker="o",
    linewidth=2
)
plt.xticks(range(1, ROUNDS + 1))
plt.xlabel("Federated round")
plt.ylabel("Estimated cumulative communication (MB)")
plt.title("EfficientNet-B0: Estimated Communication Cost")
plt.grid(True, alpha=0.3)
save_plot

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score

CURVE_DIR = os.path.join(OUTPUT_DIR, "learning_curves")
os.makedirs(CURVE_DIR, exist_ok=True)

records = []

for r in range(1, ROUNDS + 1):
    path = os.path.join(
        CHECKPOINT_DIR,
        f"efficientnet_b0_round_{r:02d}.pth"
    )

    ckpt = torch.load(
        path, map_location=DEVICE, weights_only=False
    )

    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        len(ckpt["class_names"])
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()

    result = evaluate_model(model, test_loader)

    # Read loss from the recorded training history.
    history_row = history_df.loc[
        history_df["round"] == r
    ].iloc[0]

    records.append({
        "round": r,
        "accuracy": accuracy_score(
            result["labels"], result["predictions"]
        ),
        "macro_f1": f1_score(
            result["labels"],
            result["predictions"],
            average="macro",
            zero_division=0
        ),
        "test_loss": result["loss"],
        "mean_client_query_loss":
            history_row["mean_client_query_loss"]
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

curve_df = pd.DataFrame(records)
curve_df.to_csv(
    os.path.join(CURVE_DIR, "efficientnet_b0_round_metrics.csv"),
    index=False
)

# Accuracy curve
plt.figure(figsize=(8, 5))
plt.plot(
    curve_df["round"], curve_df["accuracy"] * 100,
    marker="o", label="EfficientNet-B0"
)
plt.xlabel("Communication Round")
plt.ylabel("Test Accuracy (%)")
plt.title("Accuracy Across Communication Rounds")
plt.xticks(range(1, ROUNDS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(CURVE_DIR, "accuracy_curve.png"),
    dpi=300
)
plt.show()
plt.close()

# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(
    curve_df["round"], curve_df["test_loss"],
    marker="o", label="Global test loss"
)
plt.plot(
    curve_df["round"], curve_df["mean_client_query_loss"],
    marker="s", label="Mean client query loss"
)
plt.xlabel("Communication Round")
plt.ylabel("Cross-Entropy Loss")
plt.title("Loss Across Communication Rounds")
plt.xticks(range(1, ROUNDS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(CURVE_DIR, "loss_curve.png"),
    dpi=300
)
plt.show()
plt.close()

# Macro F1 curve
plt.figure(figsize=(8, 5))
plt.plot(
    curve_df["round"], curve_df["macro_f1"] * 100,
    marker="o", label="EfficientNet-B0"
)
plt.xlabel("Communication Round")
plt.ylabel("Macro F1-score (%)")
plt.title("Macro F1-score Across Communication Rounds")
plt.xticks(range(1, ROUNDS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(CURVE_DIR, "f1_curve.png"),
    dpi=300
)
plt.show()
plt.close()

print("Saved learning curves to:", CURVE_DIR)
display(curve_df.round(4))

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models
from google.colab import drive

# 1. Reconnect Google Drive
drive.mount("/content/drive")

# 2. Locate the previously trained checkpoint
BASE_DIR = (
    "/content/drive/MyDrive/Brain_Tumor_FL_MAML/"
    "EfficientNetB0_Retraining"
)

CHECKPOINT_PATH = os.path.join(
    BASE_DIR,
    "efficientnet_b0_federated_maml_final.pth"
)

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
ABLATION_DIR = os.path.join(
    "/content/drive/MyDrive/Brain_Tumor_FL_MAML",
    "EfficientNetB0_Ablation"
)

os.makedirs(ABLATION_DIR, exist_ok=True)

assert os.path.isfile(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

# 3. Configure device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4. Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

CLASS_NAMES = checkpoint["class_names"]
NUM_CLASSES = len(CLASS_NAMES)

# 5. Recreate the EfficientNet-B0 architecture
global_model = models.efficientnet_b0(weights=None)
global_model.classifier[1] = nn.Linear(
    global_model.classifier[1].in_features,
    NUM_CLASSES
)

global_model.load_state_dict(checkpoint["model_state_dict"])
global_model = global_model.to(DEVICE)
global_model.eval()

print("\nSaved model restored successfully.")
print("Architecture:", checkpoint["model_name"])
print("Checkpoint round:", checkpoint["round"])
print("Classes:", CLASS_NAMES)
print("Parameters:", sum(p.numel() for p in global_model.parameters()))
print("Checkpoint size (MB):",
      round(os.path.getsize(CHECKPOINT_PATH) / (1024**2), 2))
print("Ablation output directory:", ABLATION_DIR)

In [ ]:
!pip -q install kagglehub scikit-learn seaborn

import os
import random
import numpy as np
import torch
import pandas as pd

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Restore dataset
import kagglehub

dataset_path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

# Use the known dataset layout if available; otherwise use the download path
candidate_roots = [
    "/kaggle/input/brain-tumor-mri-dataset",
    dataset_path
]

DATASET_ROOT = next(
    (
        root for root in candidate_roots
        if os.path.isdir(os.path.join(root, "Training"))
        and os.path.isdir(os.path.join(root, "Testing"))
    ),
    None
)

assert DATASET_ROOT is not None, (
    f"Could not find Training and Testing folders under: {candidate_roots}"
)

TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")
TEST_DIR = os.path.join(DATASET_ROOT, "Testing")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLIENTS = 5

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    TRAIN_DIR, transform=transform
)
test_dataset = datasets.ImageFolder(
    TEST_DIR, transform=transform
)

assert train_dataset.classes == CLASS_NAMES, (
    f"Class mismatch: checkpoint={CLASS_NAMES}, "
    f"dataset={train_dataset.classes}"
)

# Recreate five balanced clients deterministically
rng = np.random.default_rng(SEED)
targets = np.array(train_dataset.targets)
client_indices = [[] for _ in range(NUM_CLIENTS)]

for class_idx in range(NUM_CLASSES):
    class_indices = np.where(targets == class_idx)[0]
    rng.shuffle(class_indices)

    for client_id, split in enumerate(
        np.array_split(class_indices, NUM_CLIENTS)
    ):
        client_indices[client_id].extend(split.tolist())

for indices in client_indices:
    rng.shuffle(indices)

client_loaders = {}
client_info = {}

for client_id, indices in enumerate(client_indices, start=1):
    subset = Subset(train_dataset, indices)

    client_loaders[f"client{client_id}"] = DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    client_info[f"client{client_id}"] = {
        "samples": len(indices),
        "reputation": 1.0,
        "last_accuracy": 0.0,
        "last_loss": float("inf"),
        "accuracy_history": [],
        "loss_history": [],
        "contribution_history": [],
        "selection_history": []
    }

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataset restored.")
print("Dataset root:", DATASET_ROOT)
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Classes:", train_dataset.classes)

print("\nClient partitions:")
for client, loader in client_loaders.items():
    labels = [
        train_dataset.targets[idx]
        for idx in loader.dataset.indices
    ]
    print(
        f"{client}: {len(labels)} images | "
        f"class counts={dict(zip(*np.unique(labels, return_counts=True)))}"
    )

print("\nTest batches:", len(test_loader))
print("Saved EfficientNet-B0 weights remain unchanged.")

In [ ]:
import os
import json
import pandas as pd

ABLATION_CONFIGS = {
    "full_model": {
        "use_llm": True,
        "use_multi_agent_decomposition": True,
        "use_reputation_tracking": True,
        "use_contribution_aware_aggregation": True,
        "description": "Full LLM-supervised multi-agent framework"
    },
    "minus_llm": {
        "use_llm": False,
        "use_multi_agent_decomposition": True,
        "use_reputation_tracking": True,
        "use_contribution_aware_aggregation": True,
        "description": "Rule-based decisions with agent decomposition"
    },
    "minus_multi_agent_decomposition": {
        "use_llm": True,
        "use_multi_agent_decomposition": False,
        "use_reputation_tracking": True,
        "use_contribution_aware_aggregation": True,
        "description": "LLM enabled without multi-agent task decomposition"
    },
    "minus_reputation_tracking": {
        "use_llm": True,
        "use_multi_agent_decomposition": True,
        "use_reputation_tracking": False,
        "use_contribution_aware_aggregation": True,
        "description": "No reputation tracking; contribution weighting uses local accuracy"
    },
    "minus_contribution_aware_aggregation": {
        "use_llm": True,
        "use_multi_agent_decomposition": True,
        "use_reputation_tracking": True,
        "use_contribution_aware_aggregation": False,
        "description": "Standard sample-weighted FedAvg instead of contribution-aware aggregation"
    }
}

config_path = os.path.join(
    ABLATION_DIR, "ablation_configurations.json"
)

with open(config_path, "w") as f:
    json.dump(ABLATION_CONFIGS, f, indent=4)

config_table = pd.DataFrame([
    {
        "Experiment": name,
        "LLM": cfg["use_llm"],
        "Multi-agent decomposition": cfg["use_multi_agent_decomposition"],
        "Reputation tracking": cfg["use_reputation_tracking"],
        "Contribution-aware aggregation": cfg["use_contribution_aware_aggregation"]
    }
    for name, cfg in ABLATION_CONFIGS.items()
])

display(config_table)
print("Configurations saved to:", config_path)
print("No training has started yet.")

In [ ]:
import importlib.util
import os
import torch

print("=== Runtime ===")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== Required libraries ===")
for package in ["torchvision", "sklearn", "pandas", "transformers", "learn2learn"]:
    print(f"{package}:",
          "Available" if importlib.util.find_spec(package) else "Not installed")

print("\n=== Required variables ===")
for name in [
    "global_model",
    "checkpoint",
    "train_dataset",
    "test_dataset",
    "client_loaders",
    "client_info",
    "test_loader",
    "CLASS_NAMES",
    "ABLATION_CONFIGS",
    "ABLATION_DIR"
]:
    print(f"{name}:",
          "Available" if name in globals() else "Missing")

print("\n=== Uploaded-code integration ===")
print("Qwen model loaded:", "llm" in globals() or "model" in globals())
print("Super-agent instance available:", "llm_super_agent" in globals())

print("\n=== Checkpoint ===")
print("Exists:", os.path.isfile(CHECKPOINT_PATH))
print("Size (MB):", round(os.path.getsize(CHECKPOINT_PATH) / (1024**2), 2)
      if os.path.isfile(CHECKPOINT_PATH) else "Unavailable")

print("\nDiagnostic only — no training performed.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading Qwen tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen model...")
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

llm_model.eval()

def ask_llm(prompt, max_new_tokens=160, temperature=0.1, top_p=0.9):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an LLM supervisor for a federated learning "
                "system classifying brain MRI images into four classes. "
                "Give concise, evidence-based responses. Follow the "
                "requested output format exactly."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

print("\nQwen loaded successfully.")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB"
      if torch.cuda.is_available() else "GPU unavailable")

print("\nInference test:")
print(ask_llm(
    "Reply with exactly this phrase: LLM_INFERENCE_OK",
    max_new_tokens=20,
    temperature=0
))

In [ ]:
import json
import re
from datetime import datetime

class LLMSuperAgent:
    def __init__(self, llm_fn):
        self.llm_fn = llm_fn
        self.round_memory = []
        self.enabled = True
        print("LLM SuperAgent initialized.")

    def _ask(self, prompt, max_new_tokens=160):
        if not self.enabled:
            return "LLM disabled for this experiment."
        try:
            return self.llm_fn(
                prompt,
                max_new_tokens=max_new_tokens,
                temperature=0
            )
        except Exception as exc:
            print("LLM inference warning:", str(exc)[:200])
            return "LLM_UNAVAILABLE"

    def assign_selection_task(self):
        return self._ask(
            "State your role in client selection in one sentence.",
            max_new_tokens=40
        )

    def select_clients(self, client_info, num_clients=3,
                       use_reputation=True):
        summaries = {}
        for client, info in client_info.items():
            acc = float(info.get("last_accuracy", 0.0))
            rep = float(info.get("reputation", 1.0))
            samples = int(info.get("samples", 0))
            score = 0.7 * acc + 0.3 * rep if use_reputation else acc
            summaries[client] = {
                "accuracy": round(acc, 4),
                "reputation": round(rep, 4) if use_reputation else None,
                "samples": samples,
                "selection_score": round(score, 4)
            }

        prompt = (
            "You are selecting clients for a federated brain MRI classifier. "
            f"Select exactly {num_clients} distinct client IDs from "
            f"{list(client_info.keys())}. Use the following metrics:\n"
            f"{json.dumps(summaries)}\n"
            "Return ONLY valid JSON in this format: "
            '{"selected_clients":["client1","client2","client3"]}.'
        )
        response = self._ask(prompt, max_new_tokens=100)

        try:
            match = re.search(r"\{.*\}", response, flags=re.S)
            parsed = json.loads(match.group(0)) if match else {}
            selected = parsed.get("selected_clients", [])
            valid = list(dict.fromkeys(
                c for c in selected if c in client_info
            ))
            if len(valid) == num_clients:
                return valid, response
        except Exception:
            pass

        # Deterministic fallback if LLM output is malformed/unavailable.
        ranked = sorted(
            summaries,
            key=lambda c: summaries[c]["selection_score"],
            reverse=True
        )
        return ranked[:num_clients], "RULE_BASED_FALLBACK: " + response

    def verify_selection(self, selected_clients):
        return self._ask(
            f"Review this selected client list: {selected_clients}. "
            "Give one concise observation.",
            max_new_tokens=60
        )

    def assign_aggregation_task(self):
        return self._ask(
            "State your role in reviewing federated aggregation in one sentence.",
            max_new_tokens=40
        )

    def verify_aggregation(self, contribution_scores):
        return self._ask(
            "Review these normalized aggregation weights: "
            f"{json.dumps(contribution_scores)}. "
            "Check whether they sum approximately to 1 and give one sentence.",
            max_new_tokens=80
        )

    def assign_evaluation_task(self):
        return self._ask(
            "State your role in reviewing model evaluation in one sentence.",
            max_new_tokens=40
        )

    def verify_evaluation(self, accuracy, precision, recall, f1):
        metrics = {
            "accuracy": float(accuracy),
            "macro_precision": float(precision),
            "macro_recall": float(recall),
            "macro_f1": float(f1)
        }
        return self._ask(
            "Review these test metrics for a four-class brain MRI classifier. "
            "Do not claim clinical readiness. Metrics: "
            f"{json.dumps(metrics)}. Give a concise interpretation.",
            max_new_tokens=100
        )

    def monitor_selection(self, selected_clients):
        return self.verify_selection(selected_clients)

    def update_round_memory(
        self, round_number, selected_clients, contribution_scores,
        accuracy, precision, recall, f1,
        selection_summary="", aggregation_summary="",
        evaluation_summary=""
    ):
        self.round_memory.append({
            "round": int(round_number),
            "selected_clients": list(selected_clients),
            "contribution_scores": {
                k: float(v) for k, v in contribution_scores.items()
            },
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "selection_summary": str(selection_summary),
            "aggregation_summary": str(aggregation_summary),
            "evaluation_summary": str(evaluation_summary)
        })

    def generate_round_summary(self):
        if not self.round_memory:
            return "No rounds recorded."
        latest = self.round_memory[-1]
        summary = self._ask(
            "Summarize this federated-learning round in 2 concise sentences. "
            "Do not invent results: " + json.dumps(latest),
            max_new_tokens=100
        )
        print("\nLLM round summary:", summary)
        return summary


llm_super_agent = LLMSuperAgent(ask_llm)

# Verify the LLM's client-selection output without starting training.
test_selection, test_selection_reason = llm_super_agent.select_clients(
    client_info,
    num_clients=3,
    use_reputation=True
)

print("\nTest selection:", test_selection)
print("LLM response / fallback:", test_selection_reason[:500])
print("\nTraining started: NO")

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

# Common experimental settings
ABLATION_SEED = 42
ABLATION_ROUNDS = 10
ABLATION_LOCAL_EPOCHS = 3
ABLATION_CLIENTS_PER_ROUND = 3
ABLATION_LR = 1e-4

random.seed(ABLATION_SEED)
np.random.seed(ABLATION_SEED)
torch.manual_seed(ABLATION_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(ABLATION_SEED)

# Validate model architecture
print("=== Model validation ===")
print("Model type:", type(global_model).__name__)
print("Classifier:", global_model.classifier)
print("Output classes:", NUM_CLASSES)
print("Checkpoint:", CHECKPOINT_PATH)

assert hasattr(global_model, "features"), (
    "Expected an EfficientNet-style model with a features module."
)
assert hasattr(global_model, "classifier"), (
    "Expected an EfficientNet classifier."
)
assert len(client_loaders) == 5, "Expected five client loaders."

# Verify class mapping and one input batch
print("\n=== Dataset validation ===")
print("Class names:", CLASS_NAMES)
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

for client_name, loader in client_loaders.items():
    print(f"{client_name}: {len(loader.dataset)} images")

images, labels = next(iter(client_loaders["client1"]))
print("\nInput batch shape:", tuple(images.shape))
print("Label batch shape:", tuple(labels.shape))
print("Labels in batch:", sorted(labels.unique().tolist()))

# Check that the model can produce four-class outputs
global_model.eval()
with torch.inference_mode():
    test_logits = global_model(images[:2].to(DEVICE))

print("Model output shape:", tuple(test_logits.shape))
assert test_logits.shape == (2, NUM_CLASSES), (
    "Model output does not match the four-class task."
)

# Preserve the same initialization for every ablation
ABLATION_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in global_model.state_dict().items()
}

print("\nShared initialization captured.")
print("Rounds:", ABLATION_ROUNDS)
print("Local epochs:", ABLATION_LOCAL_EPOCHS)
print("Clients per round:", ABLATION_CLIENTS_PER_ROUND)
print("Learning rate:", ABLATION_LR)
print("\nTraining started: NO")

In [ ]:
import copy
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

# Ensure output directory exists
os.makedirs(ABLATION_DIR, exist_ok=True)

def reset_client_state():
    """Fresh reputation and metric histories for each experiment."""
    state = {}
    for client, info in client_info.items():
        state[client] = {
            "loader": info["loader"],
            "samples": len(info["loader"].dataset),
            "reputation": 1.0,
            "last_accuracy": 0.0,
            "last_loss": float("inf"),
            "accuracy_history": [],
            "loss_history": [],
            "contribution_history": [],
            "selection_history": []
        }
    return state


def evaluate_efficientnet(model, loader=test_loader):
    model.eval()
    all_labels, all_preds = [], []
    total_loss, total_count = 0.0, 0
    criterion = nn.CrossEntropyLoss()

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_count += labels.size(0)
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())

    return {
        "loss": total_loss / max(total_count, 1),
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision_macro": precision_score(
            all_labels, all_preds, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            all_labels, all_preds, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            all_labels, all_preds, average="macro", zero_division=0
        )
    }


def train_client_efficientnet(model, loader, local_epochs=3):
    """Standard local fine-tuning; no learn2learn dependency."""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=ABLATION_LR)
    criterion = nn.CrossEntropyLoss()

    running_loss, correct, count = 0.0, 0, 0

    for epoch in range(local_epochs):
        model.train()
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += batch_size

    return {
        "accuracy": correct / max(count, 1),
        "loss": running_loss / max(count, 1)
    }


def choose_clients(config, state, num_clients=3):
    if config["use_llm"]:
        selected, response = llm_super_agent.select_clients(
            state,
            num_clients=num_clients,
            use_reputation=config["use_reputation_tracking"]
        )
        source = (
            "llm" if not response.startswith("RULE_BASED_FALLBACK")
            else "rule_based_fallback"
        )
        return selected, source, response

    def score(item):
        client, info = item
        if config["use_reputation_tracking"]:
            return (
                0.6 * info["

In [ ]:
import copy
import math

print("=== Ablation configuration checks ===")

expected_experiments = {
    "full_model",
    "minus_llm",
    "minus_multi_agent_decomposition",
    "minus_reputation_tracking",
    "minus_contribution_aware_aggregation"
}

assert set(ABLATION_CONFIGS.keys()) == expected_experiments

for name, config in ABLATION_CONFIGS.items():
    print(
        f"{name}: "
        f"LLM={config['use_llm']}, "
        f"multi-agent={config['use_multi_agent_decomposition']}, "
        f"reputation={config['use_reputation_tracking']}, "
        f"contribution-aware={config['use_contribution_aware_aggregation']}"
    )

# Test aggregation using synthetic weights; no training.
synthetic_state = {
    "client1": {"samples": 1120, "reputation": 0.9, "last_accuracy": 0.8},
    "client2": {"samples": 1120, "reputation": 0.8, "last_accuracy": 0.7},
    "client3": {"samples": 1120, "reputation": 0.7, "last_accuracy": 0.6}
}

synthetic_weights = []
for value in [1.0, 2.0, 3.0]:
    synthetic_weights.append({
        "weight": torch.tensor([value]),
        "counter": torch.tensor(1, dtype=torch.long)
    })

print("\n=== Aggregation smoke test ===")

for experiment_name in [
    "full_model",
    "minus_reputation_tracking",
    "minus_contribution_aware_aggregation"
]:
    config = ABLATION_CONFIGS[experiment_name]
    aggregated, weights = aggregate_client_models(
        synthetic_weights,
        ["client1", "client2", "client3"],
        synthetic_state,
        config
    )

    weight_sum = sum(weights.values())
    assert math.isclose(weight_sum, 1.0, rel_tol=1e-6)
    assert torch.isfinite(aggregated["weight"]).all()

    print(
        experiment_name,
        "| weights:", {k: round(v, 4) for k, v in weights.items()},
        "| sum:", round(weight_sum, 6),
        "| aggregate:", aggregated["weight"].tolist()
    )

print("\nAll smoke tests passed.")
print("No model training performed.")

In [ ]:
import os
import json
import copy
import time
import random
import numpy as np
import pandas as pd
import torch

os.makedirs(ABLATION_DIR, exist_ok=True)

def save_ablation_progress(experiment_name, model, history, state, logs):
    exp_dir = os.path.join(ABLATION_DIR, experiment_name)
    os.makedirs(exp_dir, exist_ok=True)

    # Save model and history after every completed round.
    temp_path = os.path.join(exp_dir, "latest_checkpoint.tmp")
    final_path = os.path.join(exp_dir, "latest_checkpoint.pth")

    torch.save({
        "experiment": experiment_name,
        "model_state_dict": model.state_dict(),
        "history": history,
        "client_state": {
            c: {
                k: v for k, v in info.items()
                if k != "loader"
            }
            for c, info in state.items()
        },
        "completed_rounds": len(history),
        "seed": ABLATION_SEED
    }, temp_path)
    os.replace(temp_path, final_path)

    pd.DataFrame(history).to_csv(
        os.path.join(exp_dir, "round_metrics.csv"), index=False
    )

    with open(os.path.join(exp_dir, "decision_logs.json"), "w") as f:
        json.dump(logs, f, indent=2)

    return final_path


def centralized_llm_selection(state, num_clients=3,
                               use_reputation=True):
    """Single-controller selection for the no-decomposition experiment."""
    summary = {}
    for client, info in state.items():
        summary[client] = {
            "accuracy": round(float(info["last_accuracy"]), 4),
            "reputation": (
                round(float(info["reputation"]), 4)
                if use_reputation else None
            ),
            "samples": int(info["samples"])
        }

    response = ask_llm(
        "Act as one centralized controller for federated learning. "
        f"Choose exactly {num_clients} clients from {list(state.keys())}. "
        f"Client metrics: {json.dumps(summary)}. "
        'Return only JSON: {"selected_clients":["client1","client2","client3"]}',
        max_new_tokens=100,
        temperature=0
    )

    try:
        match = re.search(r"\{.*\}", response, flags=re.S)
        parsed = json.loads(match.group(0)) if match else {}
        selected = list(dict.fromkeys(
            c for c in parsed.get("selected_clients", [])
            if c in state
        ))
        if len(selected) == num_clients:
            return selected, "centralized_llm", response
    except Exception:
        pass

    ranked = sorted(
        state,
        key=lambda c: (
            0.6 * state[c]["reputation"] +
            0.4 * state[c]["samples"] / 1120
            if use_reputation else state[c]["last_accuracy"]
        ),
        reverse=True
    )
    return ranked[:num_clients], "centralized_rule_fallback", response


def run_ablation_experiment(experiment_name, config):
    print("\n" + "=" * 75)
    print("STARTING EXPERIMENT:", experiment_name)
    print("=" * 75)

    # Each experiment starts from the same checkpoint state.
    model = copy.deepcopy(global_model).to(DEVICE)
    model.load_state_dict(ABLATION_INITIAL_STATE)
    state = reset_client_state()
    history, logs = [], []

    # Fresh per-experiment LLM memory.
    llm_super_agent.round_memory = []

    start_time = time.time()

    for round_number in range(1, ABLATION_ROUNDS + 1):
        print(f"\n[{experiment_name}] Round {round_number}/{ABLATION_ROUNDS}")

        # 1. Client selection
        if config["use_llm"] and config["use_multi_agent_decomposition"]:
            selected, source, selection_response = choose_clients(
                config, state, ABLATION_CLIENTS_PER_ROUND
            )
        elif config["use_llm"]:
            selected, source, selection_response = centralized_llm_selection(
                state,
                ABLATION_CLIENTS_PER_ROUND,
                config["use_reputation_tracking"]
            )
        else:
            selected, source, selection_response = choose_clients(
                config, state, ABLATION_CLIENTS_PER_ROUND
            )

        print("Selected:", selected, "| Source:", source)

        # 2. Local training
        local_states, local_results = [], {}
        for client in selected:
            local_model = copy.deepcopy(model).to(DEVICE)
            local_result = train_client_efficientnet(
                local_model,
                state[client]["loader"],
                local_epochs=ABLATION_LOCAL_EPOCHS
            )
            local_states.append({
                k: v.detach().clone()
                for k, v in local_model.state_dict().items()
            })
            local_results[client] = local_result
            del local_model

        # 3. Reputation / client metric updates
        update_client_reputation(
            state, selected, local_results,
            enabled=config["use_reputation_tracking"]
        )

        # 4. Aggregation
        aggregated_state, contribution_weights = aggregate_client_models(
            local_states, selected, state, config
        )
        model.load_state_dict(aggregated_state)

        # 5. Global evaluation
        metrics = evaluate_efficientnet(model, test_loader)

        # For decomposed agents, separately review the round decisions.
        if config["use_llm"] and config["use_multi_agent_decomposition"]:
            selection_review = llm_super_agent.verify_selection(selected)
            aggregation_review = llm_super_agent.verify_aggregation(
                contribution_weights
            )
            evaluation_review = llm_super_agent.verify_evaluation(
                metrics["accuracy"],
                metrics["precision_macro"],
                metrics["recall_macro"],
                metrics["f1_macro"]
            )
        else:
            selection_review = "Centralized or rule-based coordination."
            aggregation_review = "Aggregation performed by configured algorithm."
            evaluation_review = "Evaluation metrics computed programmatically."

        # Record actual component behavior for auditability.
        record = {
            "experiment": experiment_name,
            "round": round_number,
            "selected_clients": ",".join(selected),
            "selection_source": source,
            **metrics,
            "client_selection_response": selection_response,
            "selection_review": selection_review,
            "aggregation_review": aggregation_review,
            "evaluation_review": evaluation_review,
            "elapsed_seconds": round(time.time() - start_time, 2)
        }
        history.append(record)

        logs.append({
            "round": round_number,
            "selected_clients": selected,
            "selection_source": source,
            "selection_response": selection_response,
            "contribution_weights": contribution_weights,
            "local_results": local_results,
            "selection_review": selection_review,
            "aggregation_review": aggregation_review,
            "evaluation_review": evaluation_review
        })

        print(
            f"Test accuracy: {metrics['accuracy']:.4f} | "
            f"Macro-F1: {metrics['f1_macro']:.4f} | "
            f"Test loss: {metrics['loss']:.4f}"
        )

        # Save recovery artifacts before moving to the next round.
        save_ablation_progress(experiment_name, model, history, state, logs)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_metrics = evaluate_efficientnet(model, test_loader)
    exp_dir = os.path.join(ABLATION_DIR, experiment_name)
    torch.save(
        model.state_dict(),
        os.path.join(exp_dir, "final_model.pth")
    )

    result = {
        "experiment": experiment_name,
        **final_metrics,
        "completed_rounds": len(history),
        "duration_minutes": round((time.time() - start_time) / 60, 2)
    }

    with open(os.path.join(exp_dir, "final_metrics.json"), "w") as f:
        json.dump(result, f, indent=2)

    print("\nExperiment finished:", result)
    return result


print("Experiment runner defined.")
print("Checkpointing: after every round")
print("Experiment count:", len(ABLATION_CONFIGS))
print("Training started: NO")

In [ ]:
def reset_client_state():
    state = {}

    for client, info in client_info.items():
        # Get the loader from whichever dictionary contains it
        loader = info.get("loader") if isinstance(info, dict) else None

        if loader is None:
            loader = client_loaders.get(client)

        if loader is None:
            raise KeyError(
                f"No DataLoader found for {client}. "
                f"Available client_loaders: {list(client_loaders.keys())}"
            )

        state[client] = {
            "loader": loader,
            "samples": len(loader.dataset),
            "reputation": 1.0,
            "last_accuracy": 0.0,
            "last_loss": float("inf"),
            "accuracy_history": [],
            "loss_history": [],
        }

    return state


# Verify the fix before starting training
test_client_state = reset_client_state()

print("Client state initialized successfully.")
print("Number of clients:", len(test_client_state))

for client, info in test_client_state.items():
    print(f"{client}: {info['samples']} samples")

In [ ]:
full_model_result = run_ablation_experiment(
    experiment_name="full_model",
    config=ABLATION_CONFIGS["full_model"]
)

print("\n=== FULL MODEL RESULT ===")
display(pd.DataFrame([full_model_result]))

In [ ]:
minus_llm_result = run_ablation_experiment(
    experiment_name="minus_llm",
    config=ABLATION_CONFIGS["minus_llm"]
)

print("\n=== MINUS LLM RESULT ===")
display(pd.DataFrame([minus_llm_result]))

In [ ]:
minus_multi_agent_result = run_ablation_experiment(
    experiment_name="minus_multi_agent_decomposition",
    config=ABLATION_CONFIGS["minus_multi_agent_decomposition"]
)

print("\n=== MINUS MULTI-AGENT RESULT ===")
display(pd.DataFrame([minus_multi_agent_result]))

In [ ]:
minus_reputation_result = run_ablation_experiment(
    experiment_name="minus_reputation_tracking",
    config=ABLATION_CONFIGS["minus_reputation_tracking"]
)

print("\n=== MINUS REPUTATION RESULT ===")
display(pd.DataFrame([minus_reputation_result]))

In [ ]:
minus_contribution_result = run_ablation_experiment(
    experiment_name="minus_contribution_aware_aggregation",
    config=ABLATION_CONFIGS["minus_contribution_aware_aggregation"]
)

print("\n=== MINUS CONTRIBUTION-AWARE AGGREGATION RESULT ===")
display(pd.DataFrame([minus_contribution_result]))

In [ ]:
import os
import pandas as pd

print("ABLATION DIRECTORY:", ABLATION_DIR)

for experiment in ABLATION_CONFIGS:
    experiment_dir = os.path.join(ABLATION_DIR, experiment)
    metrics_path = os.path.join(experiment_dir, "round_metrics.csv")
    final_metrics_path = os.path.join(experiment_dir, "final_metrics.json")
    checkpoint_path = os.path.join(experiment_dir, "final_model.pth")

    print(f"\n--- {experiment} ---")
    print("Round metrics saved:", os.path.exists(metrics_path))
    print("Final metrics saved:", os.path.exists(final_metrics_path))
    print("Final checkpoint saved:", os.path.exists(checkpoint_path))

    if os.path.exists(metrics_path):
        df = pd.read_csv(metrics_path)
        print("Recorded rounds:", len(df))


In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Load round-by-round metrics from Google Drive
experiments = {
    "LLM (Full Model)": "full_model",
    "Rule-Based Baseline": "minus_llm",
}

round_data = {}

for label, exp_name in experiments.items():
    csv_path = os.path.join(
        ABLATION_DIR, exp_name, "round_metrics.csv"
    )

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Metrics file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"\n{label} — columns: {df.columns.tolist()}")
    round_data[label] = df

# Normalize column names to make lookup robust
def find_column(df, candidates):
    normalized = {
        str(col).strip().lower().replace(" ", "_"): col
        for col in df.columns
    }
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    raise KeyError(
        f"Could not find any of {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )

metric_candidates = {
    "accuracy": ["accuracy", "test_accuracy", "test_acc"],
    "loss": ["loss", "test_loss", "test_loss_avg"],
    "f1": ["f1_macro", "macro_f1", "f1", "test_f1"],
}

columns = {}
for label, df in round_data.items():
    columns[label] = {
        metric: find_column(df, candidates)
        for metric, candidates in metric_candidates.items()
    }

# Plot accuracy, loss, and macro-F1 across rounds
plot_metrics = [
    ("accuracy", "Test Accuracy", "Accuracy"),
    ("loss", "Test Loss", "Loss"),
    ("f1", "Macro-F1 Score", "Macro-F1"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric, title, ylabel) in zip(axes, plot_metrics):
    for label, df in round_data.items():
        x = range(1, len(df) + 1)
        y = df[columns[label][metric]].astype(float)

        ax.plot(
            x, y, marker="o", linewidth=2, markersize=5,
            label=label
        )

    ax.set_title(title)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, 11))
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.suptitle(
    "LLM Full Model vs Rule-Based Baseline",
    fontsize=15
)
plt.tight_layout()

# Save the figure to Drive
plot_path = os.path.join(
    ABLATION_DIR, "llm_vs_rule_based_learning_curves.png"
)
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nLearning curves saved to: {plot_path}")

# Summary statistics: mean and standard deviation across rounds
summary_rows = []

for label, df in round_data.items():
    row = {"Configuration": label}

    for metric, _, _ in plot_metrics:
        values = df[columns[label][metric]].astype(float)
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std(ddof=0)
        row[f"{metric}_first_round"] = values.iloc[0]
        row[f"{metric}_last_round"] = values.iloc[-1]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(4))

# Save summary
summary_path = os.path.join(
    ABLATION_DIR, "llm_vs_rule_based_curve_summary.csv"
)
summary_df.to_csv(summary_path, index=False)
print("Summary saved to:", summary_path)
